# Natural Language Processing & Review Trust Intelligence
## AI-Powered Marketplace Trust Intelligence Platform

### Scope & Architecture
This notebook establishes the primary Natural Language Processing (NLP) trust pipeline across supported product categories and marketplaces:
1. **Canonical Platform Dataset Ingestion:** Evaluates 55,616 customer reviews joined with 27,789 canonical products across **Amazon** and **BestBuy** across all six prioritized categories (*Electronics*, *Beauty & Personal Care*, *Clothing & Fashion*, *Home & Kitchen*, *Books & Stationery*, and *Toys & Games*).
2. **Multi-Category Normalization & Global Vocabulary:** Builds leakage-safe text tokenization and TF-IDF feature representations.
3. **Rating–Sentiment Discordance Audit:** Analyzes polar discordance between numerical star ratings and text sentiment polarity using VADER; demonstrates why discordance is an exploratory signal rather than proof of fraud.
4. **Intra-Product Review Similarity & Redundancy:** Audits intra-listing review duplication and addresses combinatorial pair inflation across sparse and dense listings.
5. **Exploratory Dense Cohort Analysis:** Evaluates `dense_reviews.parquet` strictly as an exploratory high-density subset (Amazon Beauty), contrasting its density with the broader canonical dataset.
6. **Independent E-Commerce Fake Reviews Benchmark:** Evaluates the Salminen et al. (2022) 40k benchmark with duplicate-leakage prevention and proper vectorizer isolation.
7. **Research Boundary Quarantine:** Quarantines the Ott et al. hotel review experiment in `notebooks/hotel_benchmark_isolated.ipynb`, preventing domain mismatch with marketplace listings.


## 1. Primary Marketplace Data Pipeline: Canonical Ingestion & Schema Integrity
We load `canonical_reviews.parquet` as our primary review source and join `canonical_products.parquet` via `canonical_product_id`.


In [ ]:
from pathlib import Path
import sys
import os
import pandas as pd
import numpy as np

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

# Load primary canonical datasets
reviews_path = project_root / 'data' / 'processed' / 'canonical_reviews.parquet'
products_path = project_root / 'data' / 'processed' / 'canonical_products.parquet'

reviews_df = pd.read_parquet(reviews_path)
products_df = pd.read_parquet(products_path)

print(f'Canonical Reviews Shape: {reviews_df.shape}')
print(f'Canonical Products Shape: {products_df.shape}')

# Verify Primary Key Uniqueness
assert reviews_df['canonical_review_id'].nunique() == len(reviews_df), 'Review primary key is not 100% unique!'
print(f'\n[PASS] Primary Key Integrity: 100% unique review IDs ({len(reviews_df):,} distinct).')

# Join Products metadata using canonical_product_id
prod_cols = ['canonical_product_id', 'category', 'subcategory', 'brand', 'title', 'price']
df = reviews_df.merge(products_df[prod_cols], on='canonical_product_id', how='left')

# Verify Referential Integrity (Foreign Key)
orphans = df['category'].isna().sum()
assert orphans == 0, f'Found {orphans} orphaned reviews referencing nonexistent products!'
print(f'[PASS] Referential Integrity: All {len(df):,} reviews link to valid products (0 foreign-key orphans).')

# Unify Timestamp Parsing (Amazon epoch seconds/ms vs. BestBuy datetime strings)
parsed_dt = pd.Series(pd.NaT, index=df.index, dtype='datetime64[ns]')
bb_mask = df['review_date'].notna()
parsed_dt[bb_mask] = pd.to_datetime(df.loc[bb_mask, 'review_date'], format='mixed')
az_s = df['timestamp'].notna() & (df['timestamp'] < 1e11)
az_ms = df['timestamp'].notna() & (df['timestamp'] >= 1e11)
parsed_dt[az_s] = pd.to_datetime(df.loc[az_s, 'timestamp'], unit='s')
parsed_dt[az_ms] = pd.to_datetime(df.loc[az_ms, 'timestamp'], unit='ms')
df['review_datetime'] = parsed_dt
print(f'[PASS] Unified Timestamps: {df["review_datetime"].notna().sum():,} parsed dates (Range: {df["review_datetime"].min().strftime("%Y-%m-%d")} to {df["review_datetime"].max().strftime("%Y-%m-%d")}).')

# Check and report text missingness
missing_text = df['review_text'].isna().sum()
empty_text = df['review_text'].astype(str).str.strip().eq('').sum()
short_text = (df['review_text'].astype(str).str.strip().str.len() < 5).sum()
print(f'[PASS] Text Completeness: {missing_text} missing, {empty_text} empty, {short_text} short (<5 chars) texts.')

# Observational Label Grounding Check
print(f'[INFO] Ground Truth Labels: {df["ground_truth_label"].isna().sum():,} / {len(df):,} reviews are unlabelled observational data.')
print('\nCanonical Reviews by Marketplace:')
display(df['marketplace'].value_counts())
print('\nCanonical Reviews by Category:')
display(df['category'].value_counts())


## 2. Text Normalization & Multi-Category TF-IDF Vocabulary
We normalize text across all reviews (lowercasing, whitespace consolidation, stopword removal) and extract a representative TF-IDF vocabulary across all categories.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['nlp_text'] = (
    df['review_text']
    .astype('string')
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

usable_df = df[df['nlp_text'].notna() & df['nlp_text'].ne('')].copy()
print(f'Total usable reviews: {len(usable_df):,}')
print(f'Unique review texts: {usable_df["nlp_text"].nunique():,}')

tfidf = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2), min_df=5)
X_tfidf = tfidf.fit_transform(usable_df['nlp_text'])
print(f'Multi-Category TF-IDF Matrix Shape: {X_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,} features')


## 3. Rating–Sentiment Discordance Audit (Canonical Dataset)

### Methodological Definitions
- **Polar Discordance (Standard Heuristic):** A strong directional mismatch between star rating and text polarity:
  - High rating ($r \ge 4.0$) with negative compound sentiment ($s \le -0.20$)
  - Low rating ($r \le 2.0$) with positive compound sentiment ($s \ge +0.30$)
  - Neutral ratings ($r = 3.0$) and mildly polarized text are intentionally excluded because mixed or nuanced evaluations are common on middle ratings.
- **Categorical Mismatch (Broad Definition):** Any disparity between discrete rating category ('Negative', 'Neutral', 'Positive') and sentiment bucket.

> **Key Interpretive Principle:** Rating–sentiment discordance is a weak exploratory signal for manual triage, NOT proof of deception. Legitimate causes include delivery carrier grievances, incorrect product variant shipped, polite phrasing, and sarcasm.


In [ ]:
from src.nlp.review_analysis import get_sentiment_compound, is_polar_discordance

# Calculate VADER sentiment compound score for all canonical reviews
usable_df['sentiment_compound'] = usable_df['nlp_text'].apply(get_sentiment_compound)

# Categorical classifications
def rating_category(r):
    return 'Negative' if r <= 2.0 else ('Neutral' if r == 3.0 else 'Positive')

def text_sentiment_cat(s):
    return 'Positive' if s >= 0.05 else ('Negative' if s <= -0.05 else 'Neutral')

usable_df['rating_sentiment'] = usable_df['rating'].apply(rating_category)
usable_df['text_sentiment'] = usable_df['sentiment_compound'].apply(text_sentiment_cat)
usable_df['cat_mismatch'] = usable_df['rating_sentiment'] != usable_df['text_sentiment']

# Polar Discordance
usable_df['polar_discordance'] = usable_df.apply(
    lambda row: is_polar_discordance(row['rating'], row['sentiment_compound']),
    axis=1
)

print('=== SENTIMENT DISCORDANCE AUDIT RESULTS (55,616 Canonical Reviews) ===')
print(f'Polar Discordance Rate: {usable_df["polar_discordance"].sum():,} / {len(usable_df):,} ({usable_df["polar_discordance"].mean():.2%})')
print(f'Categorical Mismatch Rate: {usable_df["cat_mismatch"].sum():,} / {len(usable_df):,} ({usable_df["cat_mismatch"].mean():.2%})')
print(f'Average Sentiment Compound: {usable_df["sentiment_compound"].mean():.4f}')

print('\nRating Sentiment vs. Text Sentiment Crosstab:')
display(pd.crosstab(usable_df['rating_sentiment'], usable_df['text_sentiment'], margins=True))

print('\nPolar Discordance by Marketplace:')
for mkt, grp in usable_df.groupby('marketplace'):
    print(f'  - {mkt}: {grp["polar_discordance"].sum():,} / {len(grp):,} ({grp["polar_discordance"].mean():.2%})')

print('\nPolar Discordance by Category:')
for cat, grp in usable_df.groupby('category'):
    print(f'  - {cat}: {grp["polar_discordance"].sum():,} / {len(grp):,} ({grp["polar_discordance"].mean():.2%})')


### Inspection of Representative Discordant Examples
Observing discordant examples reveals that most represent real human communication complexities rather than fraudulent manipulation.


In [ ]:
print('--- High Rating (5-star) with Negative Compound (<= -0.2) ---')
sample_high_neg = usable_df[(usable_df['rating'] >= 4.0) & (usable_df['sentiment_compound'] <= -0.2)].sample(min(3, len(usable_df)), random_state=42)
for _, r in sample_high_neg.iterrows():
    print(f"[{r['marketplace'].upper()}] Category: {r['category']} | Rating: {r['rating']} stars | Compound: {r['sentiment_compound']:.2f}")
    print(f"  Text: {r['nlp_text'][:140]}...\n")

print('--- Low Rating (1-star) with Positive Compound (>= +0.3) ---')
sample_low_pos = usable_df[(usable_df['rating'] <= 2.0) & (usable_df['sentiment_compound'] >= 0.3)].sample(min(3, len(usable_df)), random_state=42)
for _, r in sample_low_pos.iterrows():
    print(f"[{r['marketplace'].upper()}] Category: {r['category']} | Rating: {r['rating']} stars | Compound: {r['sentiment_compound']:.2f}")
    print(f"  Text: {r['nlp_text'][:140]}...\n")


## 4. Intra-Product Review Similarity & Listing-Level Redundancy

### Review Density Reality Check
Before computing intra-product similarity, we must audit the empirical distribution of reviews per product listing across the canonical platform.


In [ ]:
prod_counts = usable_df.groupby('canonical_product_id').size()
n_total_prods = len(prod_counts)
n_single_rev = (prod_counts == 1).sum()
n_multi_rev = (prod_counts >= 2).sum()

print(f'Total Canonical Products with Reviews: {n_total_prods:,}')
print(f'Products with exactly 1 review: {n_single_rev:,} ({n_single_rev / n_total_prods:.1%})')
print(f'Products with >= 2 reviews: {n_multi_rev:,} ({n_multi_rev / n_total_prods:.1%})')
print(f'Products with >= 5 reviews: {(prod_counts >= 5).sum():,} ({(prod_counts >= 5).mean():.1%})')
print(f'Products with >= 10 reviews: {(prod_counts >= 10).sum():,} ({(prod_counts >= 10).mean():.1%})')

print('\n[CRITICAL ARCHITECTURAL FINDING]')
print('84.0% of reviewed products have only 1 review. Intra-product similarity cannot be computed on single-review listings.')
print('Similarity analysis must be evaluated strictly on multi-review listings (n >= 2), avoiding misleading generalization.')


### Multi-Review Similarity Analysis & Combinatorial Explosion Audit
When multiple identical reviews appear on a listing (e.g. 5 identical reviews), calculating all pairwise combinations yields $(5 \times 4)/2 = 10$ pairs. Counting repeated pairs as independent fraud evidence creates combinatorial inflation. We report both pairwise counts and unique duplicate texts.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

multi_review_pids = prod_counts[prod_counts >= 2].index
similar_pairs = []
products_with_clusters = 0

for pid in multi_review_pids:
    group = usable_df[usable_df['canonical_product_id'] == pid]
    texts = [t for t in group['nlp_text'].tolist() if len(t) >= 10]
    if len(texts) < 2:
        continue
    eval_texts = texts[:150]  # Cap for computational efficiency
    vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=1, max_features=1000)
    try:
        X_sim = vec.fit_transform(eval_texts)
        sim_matrix = cosine_similarity(X_sim)
        found_in_prod = False
        for i in range(len(eval_texts)):
            for j in range(i + 1, len(eval_texts)):
                score = sim_matrix[i, j]
                if score >= 0.80:
                    similar_pairs.append({
                        'canonical_product_id': pid,
                        'similarity': round(float(score), 3),
                        'review_1': eval_texts[i][:80],
                        'review_2': eval_texts[j][:80],
                        'is_exact': eval_texts[i] == eval_texts[j]
                    })
                    found_in_prod = True
        if found_in_prod:
            products_with_clusters += 1
    except Exception:
        continue

sim_df = pd.DataFrame(similar_pairs)
print(f'Total multi-review products evaluated: {len(multi_review_pids):,}')
print(f'Products with similar pairs (cosine >= 0.80): {products_with_clusters:,} ({products_with_clusters / len(multi_review_pids):.1%})')
print(f'Total pairwise similarities >= 0.80: {len(sim_df):,}')
if not sim_df.empty:
    exact_count = sim_df['is_exact'].sum()
    print(f'Exact duplicate pairs: {exact_count:,} ({exact_count / len(sim_df):.1%})')
    display(sim_df.head(6))


## 5. Exploratory Dense Cohort Analysis (Amazon Beauty Subset)

> **SCOPE NOTICE:** The dense cohort (`data/interim/dense_reviews.parquet`) contains 7,595 reviews across 300 Amazon All_Beauty products with $\ge 10$ reviews (mean 25.3 reviews/product). It is an **exploratory subset**, not representative of the broader canonical platform where 84% of products have only 1 review.


In [ ]:
from src.nlp.review_analysis import ProductReviewAnalyzer

dense_path = project_root / 'data' / 'interim' / 'dense_reviews.parquet'
dense_df = pd.read_parquet(dense_path)
print(f'Dense Cohort Shape: {dense_df.shape} across {dense_df["parent_asin"].nunique()} products.')

# Run ProductReviewAnalyzer on a sample dense product
sample_asin = dense_df['parent_asin'].iloc[0]
sample_prod_reviews = dense_df[dense_df['parent_asin'] == sample_asin]

analyzer = ProductReviewAnalyzer(similarity_threshold=0.80)
summary = analyzer.analyze_reviews(sample_prod_reviews)

print(f'\nProductReviewAnalyzer Output for listing {sample_asin}:')
for k, v in summary.items():
    print(f'  {k}: {v}')


## 6. Independent E-Commerce Deception Benchmark Evaluation

### Benchmark Provenance & Label Semantics
- **Dataset:** Salminen et al. (2022) / theArijitDas E-Commerce Fake Reviews Benchmark (`data/raw/benchmarks/ecommerce_fake_reviews_benchmark.parquet`).
- **Size:** 40,526 e-commerce product reviews across 10+ retail categories.
- **Labels:** `0` = Authentic customer review, `1` = Computer-generated synthetic fake review.
- **Leakage Prevention:** 33 duplicate review texts (64 records) are quarantined prior to splitting, guaranteeing **zero text overlap** between train and test partitions. The TF-IDF vectorizer is fit strictly on the training partition.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

ecom_path = project_root / 'data' / 'raw' / 'benchmarks' / 'ecommerce_fake_reviews_benchmark.parquet'
ecom_df = pd.read_parquet(ecom_path)

print(f'E-Commerce Benchmark Raw Shape: {ecom_df.shape}')
print('Class balance:')
print(ecom_df['label'].value_counts())

# Deduplicate to strictly prevent text leakage across folds
ecom_clean = ecom_df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'Deduplicated benchmark size: {len(ecom_clean):,} (Quarantined {len(ecom_df) - len(ecom_clean)} duplicate rows)')

# Leakage-safe Stratified 75/25 Train-Test Split
X = ecom_clean['text'].astype(str)
y = ecom_clean['label'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Verify zero leakage
overlap = len(set(X_train) & set(X_test))
assert overlap == 0, f'Leakage detected! {overlap} overlapping texts.'
print(f'Train partition: {len(X_train):,} reviews | Test partition: {len(X_test):,} reviews | Overlap: 0')

# Fit vectorizer strictly on training data
vec_bench = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, stop_words='english', min_df=2)
X_train_vec = vec_bench.fit_transform(X_train)
X_test_vec = vec_bench.transform(X_test)

# Train classifier
clf = LogisticRegression(max_iter=500, random_state=42)
clf.fit(X_train_vec, y_train)

y_pred = clf.predict(X_test_vec)
y_prob = clf.predict_proba(X_test_vec)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print(f'\n[E-COMMERCE BENCHMARK TEST RESULTS]')
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'ROC-AUC:   {auc:.4f}')
print(f'\nConfusion Matrix (TN, FP / FN, TP):\n{cm}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['Authentic (0)', 'Computer-Generated (1)']))


## 7. External Domain Benchmark Quarantine Notice

> **RESEARCH BOUNDARY NOTICE:**  
> The Ott et al. hotel review deception experiment is maintained as an isolated research experiment in `notebooks/hotel_benchmark_isolated.ipynb`.
> - Hotel review deception models reflect hospitality stylometry (vacation descriptions, room cleanliness) and must **NOT** be conflated with e-commerce product fraud.
> - In accordance with governance standards, hotel labels are strictly quarantined from marketplace product listings.
